# Defining a schema

- Creating a defined schema helps with data quality and import performance. As mentioned during the lesson, we'll create a simple schema to read in the following columns:
> - Name
> - Age
> - City

- The `Name` and `City` columns are `StringType()` and the `Age` column is an `IntegerType()`.

## Instructions

- Import * from the `pyspark.sql.types` library.
- Define a new schema using the `StructType` method.
- Define a `StructField` for `name`, `age`, and `city`. Each field should correspond to the correct datatype and not be nullable.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [25]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [26]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [4]:
sc = spark.sparkContext

In [29]:
# Import the pyspark.sql.types library
from pyspark.sql.types import * # generally not a good practice to import * only import necessary modules 

# Define a new schema using the StructType method
people_schema = StructType([
  # Define a StructField for each field
  StructField('name', StringType(), False),
  StructField('age', IntegerType(), False),
  StructField('city', StringType(), True)
])



In [30]:
people_df = spark.read.format('csv').load(name='rawdata.csv', schema=people_schema)

In [31]:
people_df.show()

+----+---+----+
|name|age|city|
+----+---+----+
+----+---+----+



In [35]:
from pyspark.sql.types import *

file_path = "file:///home/talentum/UBUNTU_HOME/test-jupyter/P2/M3/SM3/3_InteractingwithDataFramesusingpysparksql/Dataset/people.csv"

pep_schema = StructType([
    StructField('id', IntegerType(), True),
    StructField('person_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('sex', StringType(), True),
    StructField('date of birth', StringType(), True)
])

people1_df = spark.read.format('csv').option("header", "true").load(path= file_path, schema = pep_schema)

people1_df.show()



+---+---------+-----------------+------+-------------+
| id|person_id|             name|   sex|date of birth|
+---+---------+-----------------+------+-------------+
|  0|      100|   Penelope Lewis|female|   1990-08-31|
|  1|      101|    David Anthony|  male|   1971-10-14|
|  2|      102|        Ida Shipp|female|   1962-05-24|
|  3|      103|     Joanna Moore|female|   2017-03-10|
|  4|      104|   Lisandra Ortiz|female|   2020-08-05|
|  5|      105|    David Simmons|  male|   1999-12-30|
|  6|      106|    Edward Hudson|  male|   1983-05-09|
|  7|      107|     Albert Jones|  male|   1990-09-13|
|  8|      108| Leonard Cavender|  male|   1958-08-08|
|  9|      109|   Everett Vadala|  male|   2005-05-24|
| 10|      110| Freddie Claridge|  male|   2002-05-07|
| 11|      111|Annabelle Rosseau|female|   1989-07-13|
| 12|      112|    Eulah Emanuel|female|   1976-01-19|
| 13|      113|       Shaun Love|  male|   1970-05-26|
| 14|      114|Alejandro Brennan|  male|   1980-12-22|
| 15|     

In [36]:
people1_df_sub = people1_df.select('name', 'sex')

people1_df_sub.write.format('csv').save('people1_csv')
# HERE he data folder people1_csv will be created on hdfs and inside it we will have data stored in no of files = no of columns we selected above
# CSV file format has a problem it doesnt store schema we have to create a intermediate table schema like 'pc' here

pc = StructType([
    StructField('name', StringType(), True),
    StructField('sex', StringType(), True),
])

df = spart.read.format('csv').option("header","true").load(path = 'people1_csv', schema = pc)

people1_df.show()

+---+---------+-----------------+------+-------------+
| id|person_id|             name|   sex|date of birth|
+---+---------+-----------------+------+-------------+
|  0|      100|   Penelope Lewis|female|   1990-08-31|
|  1|      101|    David Anthony|  male|   1971-10-14|
|  2|      102|        Ida Shipp|female|   1962-05-24|
|  3|      103|     Joanna Moore|female|   2017-03-10|
|  4|      104|   Lisandra Ortiz|female|   2020-08-05|
|  5|      105|    David Simmons|  male|   1999-12-30|
|  6|      106|    Edward Hudson|  male|   1983-05-09|
|  7|      107|     Albert Jones|  male|   1990-09-13|
|  8|      108| Leonard Cavender|  male|   1958-08-08|
|  9|      109|   Everett Vadala|  male|   2005-05-24|
| 10|      110| Freddie Claridge|  male|   2002-05-07|
| 11|      111|Annabelle Rosseau|female|   1989-07-13|
| 12|      112|    Eulah Emanuel|female|   1976-01-19|
| 13|      113|       Shaun Love|  male|   1970-05-26|
| 14|      114|Alejandro Brennan|  male|   1980-12-22|
| 15|     

In [42]:
people1_df_sub = people1_df.select('name', 'sex')

people1_df_sub.write.format('parquet').save('people1_parquet') # run it only once and commentout it when running this script second time



df = spark.read.format('parquet').load('people1_parquet')
df.groupBy('sex').count().show()
df.show()
                                            
#people1_df.show()

+------+-----+
|   sex|count|
+------+-----+
|  null| 1920|
|female|49014|
|  male|49066|
+------+-----+

+-----------------+------+
|             name|   sex|
+-----------------+------+
|   Penelope Lewis|female|
|    David Anthony|  male|
|        Ida Shipp|female|
|     Joanna Moore|female|
|   Lisandra Ortiz|female|
|    David Simmons|  male|
|    Edward Hudson|  male|
|     Albert Jones|  male|
| Leonard Cavender|  male|
|   Everett Vadala|  male|
| Freddie Claridge|  male|
|Annabelle Rosseau|female|
|    Eulah Emanuel|female|
|       Shaun Love|  male|
|Alejandro Brennan|  male|
|Robert Mcreynolds|  male|
|   Carla Spickard|female|
|Florence Eberhart|female|
|     Tina Gaskins|female|
| Florence Mulhern|female|
+-----------------+------+
only showing top 20 rows



In [41]:
%%sh 
hdfs dfs -ls

Found 10 items
drwxr-xr-x   - talentum supergroup          0 2026-10-25 17:09 .sparkStaging
drwxr-xr-x   - talentum supergroup          0 2026-10-23 14:35 2010_orders
-rw-r--r--   1 talentum supergroup      46138 2026-09-08 11:17 constitution.txt
drwxr-xr-x   - talentum supergroup          0 2026-10-21 15:01 grep_output
drwxr-xr-x   - talentum supergroup          0 2026-10-21 18:07 inverted
drwxr-xr-x   - talentum supergroup          0 2026-10-25 17:36 people1_csv
drwxr-xr-x   - talentum supergroup          0 2026-10-25 17:52 people1_parquet
drwxr-xr-x   - talentum supergroup          0 2026-10-23 14:35 software
-rw-r--r--   1 talentum supergroup    3613198 2026-10-21 16:53 stocks.csv
drwxr-xr-x   - talentum supergroup          0 2026-10-19 21:20 wordcount_output


In [16]:
import pyspark.sql.functions as F


df = spark.read.text('file:///home/talentum/hdp/pigandhive/labs/grep/constitution.txt')
df = df.select(F.explode(F.split(F.col("value")," ")).alias("words"))
print(df.printSchema())

df = df.groupBy('words').count()



df.show(20,truncate=False)



root
 |-- words: string (nullable = true)

None
+------------+-----+
|words       |count|
+------------+-----+
|Inhabitant  |3    |
|those       |6    |
|Affirmation.|1    |
|some        |1    |
|Office.     |2    |
|Legislation |1    |
|thereof.    |3    |
|highest     |3    |
|Union;      |1    |
|parts       |1    |
|construed   |4    |
|indictment  |1    |
|speedy      |1    |
|jurisdiction|3    |
|beverage    |1    |
|Day.        |1    |
|coin        |2    |
|be,         |1    |
|equal       |6    |
|Services    |1    |
+------------+-----+
only showing top 20 rows



In [7]:
%%sh
wc -l /home/talentum/hdp/pigandhive/labs/grep/constitution.txt

649 /home/talentum/hdp/pigandhive/labs/grep/constitution.txt
